In [17]:
import os
import polars as pl 
from openai import OpenAI

In [18]:
client = OpenAI(
    base_url=os.getenv("LMSTUDIO_BASE_URL", "http://localhost:1234/v1"),
    api_key=os.getenv("LMSTUDIO_API_KEY", "lm-studio")
)

In [19]:
def narrar_10_segundos(batch, previous, client = client, model_narrator ="qwen/qwen3.5-9b"):
    contexto = "\n".join(f"- {frase}" for frase in batch if frase and frase.strip())
    last = previous.strip()
    prompt = f"""
Esta es la frase anterior que debes continuar:

{last}

Estas son frases cortas que describen una secuencia de vídeo:

{contexto}

Eres un narrador de fútbol en directo para El Clásico (Real Madrid vs FC Barcelona).
Tu trabajo: convertir observaciones visuales en UNA sola frase oral, clara y dinámica.

Reglas obligatorias:
- Español (España), tono de retransmisión deportiva, presente narrativo.
- Exactamente UNA frase. Máximo 22 palabras (~10 segundos al leer en voz alta).
- Usa al menos 1 dato concreto de las observaciones: jugador, equipo, zona del campo o acción.
- Narra la EVOLUCIÓN del tramo: qué pasa al inicio → qué cambia al final. No describas solo un instante fijo.
- Si hay frase anterior: enlaza con ella, pero NO repitas sus palabras clave ni reescribas la misma idea.
- Prohibido: clichés vacíos ("gigantes del fútbol", "duelo táctico", "intenso clásico") sin acción concreta.
- Prohibido: meta-comentarios ("en este tramo", "como vemos", "la escena muestra").
- Devuelve SOLO la frase, sin comillas ni explicación.""".strip()

    response = client.responses.create(
        model=model_narrator,
        input=prompt,
        #max_output_tokens=85 
        #Comented max for testing pourpouse 
    )

    return response.output_text.strip()

In [20]:
def narracion(generated_description, model_narrator):
    frases_narrador = []
    last_frase = ""
    for i in range(0, len(generated_description), 7):
        batch = generated_description[i:i+7]
        frase = narrar_10_segundos(batch,last_frase, client, model_narrator)
        last_frase = frase
        frases_narrador.append(frase)
        print(f"Batch {i//7 + 1}: {frase}")
    return frases_narrador
# Mofication of the original function to analize 7 batches at a time that are equivalent to around 10 seconds(Desired narracion time per batch)

In [21]:
First_minute = pl.read_csv("data/Description_List_1min.csv")
First_minute.shape
# 125 analized frames (1 min) analized in batches of 3 images

(42, 1)

In [22]:
print(type(First_minute[:,0][0]),type(First_minute[0][0])) 
#example of some of the changes in polars and pandas. 
print(First_minute[:,0][0],First_minute[0][0]) 
#wile pandas directly returns the value stored, polars returns a dataframe of shape(1,1)
#the first one is de equivalent of pandas. 

<class 'str'> <class 'polars.dataframe.frame.DataFrame'>
Se trata de una fase de mediocampo muy intensa y equilibrada en el partido. Xavi Hernández e Iniesta están controlando la posesión del Barça en el centro, mientras que Real Madrid espera con Cristiano Ronaldo y Gareth Bale listos para aprovechar transiciones rápidas por las bandas. shape: (1, 1)
┌─────────────────────────────────┐
│ column_0                        │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ Se trata de una fase de medioc… │
└─────────────────────────────────┘


In [23]:
First_minute = First_minute[:,0].to_list()
Frases = narracion(First_minute, model_narrator= "google/gemma-4-e4b")


Batch 1: Mientras Xavi mantiene la posesión en el mediocampo central, Real Madrid busca desesperadamente explotar con Cristiano y Di María por el flanco derecho.
Batch 2: Aunque Di María busca por la banda derecha, Barcelona cambia el juego hacia el centro con Xavi e Iniesta, forzando al Real Madrid a reorganizar su estructura defensiva.
Batch 3: El Barça mantiene la presión central gracias a Messi y Fàbregas, forzando al Madrid a replegarse en una estructura defensiva más profunda ante la escasez de tiempo.
Batch 4: Pero ahora el Real Madrid busca explotar la amplitud, dejando que Dani Carvajal se desmarque por la banda y fuerce al Barça a replegarse hacia las laterales.
Batch 5: Pero ahora el ataque se centra en Ángel Di María (#22), que penetra por el eje central para romper la línea defensiva azul y buscar desbordar.
Batch 6: Aunque Di María amenaza el centro, la acción se desplaza al mediocampo; Messi y Xavi reorganizan buscando verticalidad por banda para superar la presión blanc

In [24]:
Frases_df = pl.DataFrame(Frases)
Frases_df.write_csv("data/Narration_1.csv")
